# RTK Wave Buoy — Time Series Figures
Publication figures: tidal record, wave displacement snapshot, orbital view.

In [ ]:
import subprocess, sys
for pkg in ['pyubx2', 'contextily', 'pyproj']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependencies OK')

In [ ]:
import sys
sys.path.insert(0, '../ubx_parsers')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
from datetime import datetime, timezone
from v3_ubx_parser import parse_ubx_file

plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'figure.dpi': 120,
})

RTK_NAMES  = {0: 'None', 1: 'Float', 2: 'Fix'}
RTK_COLORS = {0: '#aaaaaa', 1: '#f0a500', 2: '#2ca02c'}

## Load and clean data

In [ ]:
positions = parse_ubx_file('dataLog00072_moored.ubx')

records = []
for p in positions:
    records.append({
        't':                p['timestamp'],   # iTOW seconds — always valid
        'year':             p.get('year', 0),
        'month':            p.get('month', 0),
        'day':              p.get('day', 0),
        'hour':             p.get('hour', 0),
        'minute':           p.get('minute', 0),
        'second':           p.get('second', 0),
        'latitude':         p['latitude'],
        'longitude':        p['longitude'],
        'altitude_msl':     p['altitude_msl'],
        'carrier_solution': p['carrier_solution'],
    })

df = pd.DataFrame(records).sort_values('t').reset_index(drop=True)

# RTK Fix only, drop altitude spikes
df = df[(df['carrier_solution'] == 2) & (df['altitude_msl'].abs() < 5)].copy().reset_index(drop=True)

# find first record with a valid UTC time from NAV-PVT
anchor = df[df['year'] > 2000].iloc[0]
anchor_utc = datetime(int(anchor.year), int(anchor.month), int(anchor.day),
                      int(anchor.hour), int(anchor.minute), int(anchor.second),
                      tzinfo=timezone.utc)
anchor_t   = anchor['t']

# all timestamps as UTC using iTOW offset from anchor
df['dt'] = pd.to_datetime(
    [anchor_utc + pd.Timedelta(seconds=float(t - anchor_t)) for t in df['t']],
)
df = df.set_index('dt')

# ENU displacement from mean (metres)
lat0 = df['latitude'].mean()
lon0 = df['longitude'].mean()
alt0 = df['altitude_msl'].mean()
df['E'] = (df['longitude'] - lon0) * 111_320.0 * np.cos(np.radians(lat0))
df['N'] = (df['latitude']  - lat0) * 111_320.0
df['U'] =  df['altitude_msl'] - alt0

dt_diff = df['t'].diff().dropna()
print(f"Records  : {len(df)}")
print(f"Period   : {df.index[0]}  →  {df.index[-1]}")
print(f"Duration : {(df.index[-1]-df.index[0]).total_seconds()/3600:.1f} hours")
print(f"Median Δt: {dt_diff.median():.3f} s  (~{1/dt_diff.median():.1f} Hz)")
print(f"U range  : {df['U'].min():.3f} to {df['U'].max():.3f} m")

## Figure 1 — Full tidal record (48 h)
Raw 1 Hz altitude (light) + 10-minute running median (bold) to show tidal signal.

In [ ]:
tide_smooth = df['U'].rolling(window=600, center=True, min_periods=30).median()

fig, ax = plt.subplots(figsize=(12, 4))

ax.scatter(df.index, df['U'], s=0.5, c='steelblue', alpha=0.25,
           linewidths=0, label='1 Hz RTK Fix', rasterized=True)
ax.plot(df.index, tide_smooth, color='navy', linewidth=1.5,
        label='10-min median', zorder=3)
ax.axhline(0, color='k', linewidth=0.6, linestyle='--', alpha=0.4)

ax.set_xlabel('Date / Time (UTC)')
ax.set_ylabel('Vertical displacement (m)')
ax.set_title('Tidal Record — RTK GNSS Wave Buoy, Scripps Pier (June 2026)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=range(0,24,6)))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.savefig('fig_tide.png', dpi=300, bbox_inches='tight')
print('Saved fig_tide.png')
plt.show()

## Figure 2 — Wave snapshot (30 min, E / N / U)
Pick a 30-minute window with continuous RTK Fix. Linear-detrend each component to remove residual tidal slope.

In [ ]:
from scipy.signal import detrend

# 30-min window from the middle of the record
mid   = df.index[len(df) // 2]
snap  = df[mid - pd.Timedelta(minutes=15) : mid + pd.Timedelta(minutes=15)].copy()

snap['E'] = detrend(snap['E'].values)
snap['N'] = detrend(snap['N'].values)
snap['U'] = detrend(snap['U'].values)

print(f"{len(snap)} points  |  {snap.index[0]} → {snap.index[-1]}")

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(snap.index, snap['E'], color='royalblue',  lw=0.8)
axes[1].plot(snap.index, snap['N'], color='seagreen',   lw=0.8)
axes[2].plot(snap.index, snap['U'], color='tomato',     lw=0.8)

for ax, label in zip(axes, ['East (m)', 'North (m)', 'Up (m)']):
    ax.set_ylabel(label)
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.grid(True, alpha=0.3)

axes[0].set_title(f'Wave Displacement — {snap.index[0].strftime("%Y-%m-%d %H:%M")} UTC')
axes[2].set_xlabel('Time (UTC)')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[2].xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.savefig('fig_wave_snapshot.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3 — Orbital view (E vs N, colored by U)
Shows the wave orbital geometry from the same 30-min snapshot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

E_cm = snap['E'] * 100
N_cm = snap['N'] * 100
U_cm = snap['U'] * 100
t_s  = (snap.index - snap.index[0]).total_seconds().values

# --- left: E vs N (plan view) ---
ax = axes[0]
ax.plot(E_cm, N_cm, color='steelblue', lw=0.5, alpha=0.5, zorder=1)
sc = ax.scatter(E_cm, N_cm, c=U_cm, cmap='RdBu_r',
                s=8, linewidths=0, zorder=2)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Up (cm)')
ax.set_xlabel('East (cm)')
ax.set_ylabel('North (cm)')
ax.set_title('Plan view (E–N)')
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
ax.grid(True, alpha=0.3)

# --- right: U vs horizontal magnitude ---
ax2 = axes[1]
horiz = np.sqrt(E_cm**2 + N_cm**2)
ax2.plot(horiz, U_cm, color='steelblue', lw=0.5, alpha=0.5, zorder=1)
sc2 = ax2.scatter(horiz, U_cm, c=t_s, cmap='viridis',
                  s=8, linewidths=0, zorder=2)
cbar2 = plt.colorbar(sc2, ax=ax2)
cbar2.set_label('Elapsed time (s)')
ax2.set_xlabel('Horizontal displacement (cm)')
ax2.set_ylabel('Vertical displacement (cm)')
ax2.set_title('Orbital cross-section')
ax2.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
ax2.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
ax2.grid(True, alpha=0.3)

fig.suptitle(f'Wave Orbital Geometry — {snap.index[0].strftime("%Y-%m-%d %H:%M")} UTC', fontsize=11)
plt.tight_layout()
plt.savefig('fig_orbital.png', dpi=300, bbox_inches='tight')
print('Saved fig_orbital.png')
plt.show()

## Figure 4 — Power Spectral Density (wave band)
Middle 24 h of moored record. Welch method: 30-min Hann-windowed segments, 50% overlap, linearly detrended per segment. Wave band: 2–30 s period.

In [ ]:
from scipy.signal import detrend as sig_detrend

# ── middle 24 h ──────────────────────────────────────────────────────────────
t_center = df.index[0] + (df.index[-1] - df.index[0]) / 2
df_24h   = df[t_center - pd.Timedelta(hours=12) : t_center + pd.Timedelta(hours=12)]
print(f"24-h window: {df_24h.index[0]}  →  {df_24h.index[-1]}")

# ── resample to regular 1 Hz grid, interpolate short gaps ────────────────────
fs = 1.0
s = df_24h['U'].resample('1s').mean()
s = s.interpolate(method='time', limit=30)
s = s.values

# ── manual Welch: 30-min segments, 50% overlap ───────────────────────────────
chunk_len = 1800
step      = 900
win       = np.hanning(chunk_len)
win_corr  = (win ** 2).sum()

psds = []
for i in range(0, len(s) - chunk_len, step):
    chunk = s[i : i + chunk_len].copy()
    if np.isnan(chunk).mean() > 0.05:
        continue
    chunk = np.where(np.isnan(chunk), np.nanmean(chunk), chunk)
    chunk = sig_detrend(chunk, type='linear')
    chunk *= win
    fft_mag = np.abs(np.fft.rfft(chunk)) ** 2
    psds.append(2 * fft_mag / (fs * win_corr))

freqs    = np.fft.rfftfreq(chunk_len, d=1.0 / fs)
mean_psd = np.mean(psds, axis=0)
print(f"Valid segments: {len(psds)}")

# ── plot ──────────────────────────────────────────────────────────────────────
f_lo, f_hi = 1/30, 1/2

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(freqs[1:], mean_psd[1:], color='tomato', lw=1.2)

ax.axvspan(f_lo, f_hi, alpha=0.08, color='gold', label='Wave band (2–30 s)')
ax.axvline(f_lo, color='gray', lw=0.8, ls='--')
ax.axvline(f_hi, color='gray', lw=0.8, ls='--')

ax.set_xlim(f_lo * 0.8, f_hi * 1.05)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (m² / Hz)')
ax.set_title('Vertical Displacement PSD — Middle 24 h, Welch (30-min segments, 50% overlap)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# period axis on top
ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
period_ticks = [2, 3, 5, 8, 10, 15, 20, 30]
ax2.set_xticks([1/T for T in period_ticks])
ax2.set_xticklabels([str(T) for T in period_ticks])
ax2.set_xlabel('Period (s)')

plt.tight_layout()
plt.savefig('fig_psd.png', dpi=300, bbox_inches='tight')
print('Saved fig_psd.png')
plt.show()